# 05 — Final Dataset Assembly & Data Quality Validation

**IRS Form 990 financial anomaly / self-reported significant diversion project**

**Purpose:** Assemble a reproducible, auditable **full Form 990 organization–tax-year** modeling table, reconcile financial statements, and retain other 990-series returns as **separate, linkable supplementary data**. This notebook does **not** treat 990-EZ, 990-PF, or 990-T as equivalent to full Form 990 and does **not** label them as negative diversion cases.

**Run sequentially.** Start with `WRITE_OUTPUTS = False`; review audit tables, then set to `True` and rerun the final save cell. Financial mismatches are *review flags*, not allegations. The diversion label is **self-reported**, not confirmed fraud.

### Deliverables
- `data/analytics/modeling/05_form990_modeling_organization_year_v1.parquet`
- `data/analytics/modeling/05_other_series_canonical_v1.parquet` (separate supplementary records)
- `data/analytics/modeling/05_other_series_link_summary_v1.parquet` (aggregate link information only; no 990-T financial contamination)
- CSV/JSON audit reports in `reports/modeling_05/`.


## 1. Setup and input manifest

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

here = Path.cwd().resolve()
ROOT = next((p for p in (here, *here.parents) if (p / ".gitignore").exists()), None)
if ROOT is None:
    ROOT = here.parent if here.name == "notebooks" else here
CLEANED = ROOT / "data/cleaned"
ANALYTICS = ROOT / "data/analytics"
MODEL = ANALYTICS / "modeling"
REPORT = ROOT / "reports/modeling_05"
MODEL.mkdir(parents=True, exist_ok=True)
REPORT.mkdir(parents=True, exist_ok=True)

CANDIDATES = CLEANED / "form990_model_candidates_2023_2025.parquet"
CANONICAL = CLEANED / "form990_organization_year_provisional_2023_2025.parquet"
FULL_FEATURES = ANALYTICS / "financial_features/form990_canonical_organization_year_financial_features_v4.parquet"
OTHER_ROOT = ANALYTICS / "other_990_series_features"
OTHER = {form: OTHER_ROOT / f"{form.lower()}_filing_features_v1.parquet"
         for form in ("IRS990EZ", "IRS990PF", "IRS990T")}
WRITE_OUTPUTS = False  # Change to True only after reviewing Sections 2–9
BALANCE_TOLERANCE = 1.0

inputs = {"model_candidates": CANDIDATES, "canonical": CANONICAL,
          "full_990_financial": FULL_FEATURES, **OTHER}
for name, path in inputs.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")
manifest = pd.DataFrame([{"input": k, "path": str(p),
                          "rows": pq.ParquetFile(p).metadata.num_rows,
                          "columns": len(pq.ParquetFile(p).schema_arrow.names)}
                         for k,p in inputs.items()])
print("Project root:", ROOT)
display(manifest)


## 2. Inspect actual schemas and load full 990 organization-year data

**Join policy:** Notebook 02’s candidate set defines eligible labels and excludes conflicting EIN–tax-year labels. Notebook 04’s canonical financial file supplies financial fields. Join on the **selected filing ID**, and check that the EIN and tax year agree; never silently join financials from another filing of the same organization.

In [ ]:
candidates = pd.read_parquet(CANDIDATES)
canonical = pd.read_parquet(CANONICAL)
full = pd.read_parquet(FULL_FEATURES)
for name, df in [("candidates", candidates), ("canonical", canonical), ("full_features", full)]:
    print(f"\n{name}: {len(df):,} rows / {len(df.columns)} columns")
    print("Identity:", [c for c in ["filing_id", "ein", "tax_year", "fraud_label", "has_conflicting_labels"] if c in df.columns])
    print("First 30 columns:", df.columns.tolist()[:30])
assert {"ein", "tax_year", "filing_id", "fraud_label"}.issubset(candidates.columns)
assert {"ein", "tax_year", "filing_id"}.issubset(full.columns)
assert candidates["filing_id"].notna().all() and full["filing_id"].notna().all()
assert not candidates.duplicated(["ein", "tax_year"]).any(), "Candidate keys are not unique"
assert candidates["filing_id"].is_unique, "Candidate selected filing IDs are not unique"
assert full["filing_id"].is_unique, "Full financial selected filing IDs are not unique"
if "has_conflicting_labels" in candidates:
    assert not candidates["has_conflicting_labels"].fillna(False).any()


## 3. Join selected full 990 filings to their exact financial features

In [ ]:
def normalize_ein(s):
    return s.astype("string").str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True).str.zfill(9)

a = candidates.copy()
f = full.copy()
for df in (a, f):
    df["filing_id"] = df["filing_id"].astype("string")
    df["ein"] = normalize_ein(df["ein"])
    df["tax_year"] = pd.to_numeric(df["tax_year"], errors="coerce").astype("Int64")
assert a["ein"].str.fullmatch(r"\d{9}").fillna(False).all()

# Keep Notebook 02 as authoritative for identity/label metadata.
# Avoid duplicate financial columns already in Notebook 02.
join_cols = ["filing_id", "ein", "tax_year"]
feature_cols = [c for c in f.columns if c not in a.columns and c not in join_cols]
# Preserve all financial columns when names overlap via explicit prefix.
overlap = [c for c in f.columns if c in a.columns and c not in join_cols]
financial = f[join_cols + feature_cols + overlap].rename(columns={c: f"financial_{c}" for c in overlap})
model = a.merge(financial, on="filing_id", how="left", validate="one_to_one", indicator="_financial_join")
matched = model["_financial_join"].eq("both")
print("Financial join:", model["_financial_join"].value_counts(dropna=False).to_dict())
assert matched.all(), "Some candidate filing IDs lack financial records; investigate before proceeding"
# EIN and tax year must match for the exact selected filing ID.
for k in ["ein", "tax_year"]:
    left, right = f"{k}_x", f"{k}_y"
    assert left in model and right in model, f"Missing join verification columns for {k}"
    assert model[left].eq(model[right]).fillna(False).all(), f"Selected filing ID has conflicting {k}"
    model[k] = model[left]
    model = model.drop(columns=[left, right])
model = model.drop(columns="_financial_join")
assert len(model) == len(candidates)
assert not model.duplicated(["ein", "tax_year"]).any()
print(f"Assembled {len(model):,} full Form 990 organization-years")


## 4. Financial reconciliation and missingness flags

Do **not** replace missing liabilities with zero. A reported zero is distinct from an absent XML value. Revenue–expenses and assets–liabilities are evaluated only where all operands exist. Some statements can differ for legitimate accounting reasons; retain residuals for review.

In [ ]:
def numeric(col):
    return pd.to_numeric(model[col], errors="coerce") if col in model else pd.Series(np.nan, index=model.index, dtype="float64")

def residual_check(name, left, right, expected):
    residual = numeric(left) - numeric(right) - numeric(expected)
    model[name + "_residual"] = residual
    model[name + "_mismatch_flag"] = residual.notna() & residual.abs().gt(BALANCE_TOLERANCE)
    return {"check": name, "eligible": int(residual.notna().sum()),
            "mismatches_gt_tolerance": int(model[name + "_mismatch_flag"].sum()),
            "mismatch_rate_pct": round(100 * model[name + "_mismatch_flag"].sum() / max(residual.notna().sum(),1), 3)}

checks = []
if {"assets_eoy", "liabilities_eoy", "net_assets_eoy"}.issubset(model):
    checks.append(residual_check("full990_balance_sheet", "assets_eoy", "liabilities_eoy", "net_assets_eoy"))
else:
    print("Missing expected full-990 balance sheet columns; inspect Section 2")
model["full990_revenue_minus_expenses"] = numeric("revenue_cy") - numeric("expenses_cy")
model["full990_missing_core_count"] = pd.concat(
    [numeric(x).isna() for x in ["revenue_cy", "expenses_cy", "assets_eoy", "liabilities_eoy"]], axis=1
).sum(axis=1)
model["full990_negative_assets_flag"] = numeric("assets_eoy").lt(0).fillna(False)
model["full990_negative_revenue_flag"] = numeric("revenue_cy").lt(0).fillna(False)
model["full990_liabilities_missing_flag"] = numeric("liabilities_eoy").isna()
model["full990_liabilities_reported_zero_flag"] = numeric("liabilities_eoy").eq(0).fillna(False)
reconciliation = pd.DataFrame(checks)
display(reconciliation)
print("Full 990 missing core financial field distribution:")
display(model["full990_missing_core_count"].value_counts().sort_index().rename_axis("missing_fields").to_frame("records"))


## 5. Review full 990 label distribution, repeated EINs, and leakage

Labels come **only** from the full 990 candidate dataset. `fraud_label` is the existing Notebook 02 field name but denotes a **self-reported significant diversion disclosure**, not established fraud. No imputation of unknown labels. Do not use disclosure fields, label-related metadata, or same-return post-disclosure audit flags as predictors.

In [ ]:
print("Self-reported diversion label counts (including missing):")
display(model["fraud_label"].value_counts(dropna=False).rename_axis("label").to_frame("records"))
print("Distinct EINs:", model["ein"].nunique())
print("EINs appearing in multiple tax years:", int(model.groupby("ein")["tax_year"].nunique().gt(1).sum()))
print("Tax year distribution:")
display(model["tax_year"].value_counts(dropna=False).sort_index().rename_axis("tax_year").to_frame("records"))

# Candidate feature names only: not a training split, no learned imputation, no target encoding.
excluded_exact = {"fraud_label", "raw_diversion_response", "ever_positive", "positive_filing_count",
                  "has_conflicting_labels", "distinct_labels", "filing_count", "has_multiple_filings",
                  "selection_verified_amendment", "selection_method", "source_xml", "archive_name",
                  "organization_name", "filing_id", "ein", "tax_year"}
excluded_patterns = ("diversion", "fraud", "label", "_status", "_source_path", "_mismatch_flag", "_review_flag", "_missing_flag")
numeric_candidates = [c for c in model.select_dtypes(include="number").columns
                      if c not in excluded_exact and not any(x in c.lower() for x in excluded_patterns)]
print("Numeric columns requiring feature review:", len(numeric_candidates))
print(numeric_candidates[:50])
print("NOTE: this is NOT an approved feature list. Verify prediction-time availability and leakage before Notebook 06.")


## 6. Load supplementary 990-EZ / 990-PF / 990-T records separately

**Do not append these as negative training rows.** These returns serve different purposes; Form 990-T may be filed in addition to another return. We preserve filing-level provenance and quarantine invalid keys. Canonicalization is within each **form type + EIN + tax year**, not across form types. It is deterministic, but **not a verified amendment preference**.

In [ ]:
others = {}
other_audit = []
for form, path in OTHER.items():
    df = pd.read_parquet(path)
    assert {"filing_id", "ein", "tax_year"}.issubset(df.columns), f"Missing identity in {form}"
    df["filing_id"] = df["filing_id"].astype("string")
    df["ein"] = normalize_ein(df["ein"])
    df["tax_year"] = pd.to_numeric(df["tax_year"], errors="coerce").astype("Int64")
    df["form_type"] = form
    df["key_valid"] = df["ein"].str.fullmatch(r"\d{9}").fillna(False) & df["tax_year"].notna()
    assert df["filing_id"].is_unique, f"Repeated filing IDs in {form}"
    other_audit.append({"form_type":form, "filings":len(df),
                        "invalid_keys":int((~df["key_valid"]).sum()),
                        "unique_valid_org_year":int(df.loc[df.key_valid, ["ein","tax_year"]].drop_duplicates().shape[0]),
                        "duplicated_valid_org_year_filings":int(df.loc[df.key_valid].duplicated(["ein","tax_year"], keep=False).sum())})
    others[form] = df
other_counts = pd.DataFrame(other_audit)
display(other_counts)


## 7. Supplementary reconciliation and conservative canonicalization

In [ ]:
def as_num(df, field):
    return pd.to_numeric(df[field], errors="coerce") if field in df else pd.Series(np.nan, index=df.index)

def reconcile(df, name, a, b, c):
    residual = as_num(df,a) - as_num(df,b) - as_num(df,c)
    df[name + "_residual"] = residual
    df[name + "_mismatch_flag"] = residual.notna() & residual.abs().gt(BALANCE_TOLERANCE)
    return {"form_type":df.form_type.iloc[0], "check":name,
            "eligible":int(residual.notna().sum()),
            "mismatch":int(df[name + "_mismatch_flag"].sum())}

supp_checks = []
ez = others["IRS990EZ"]
supp_checks.append(reconcile(ez, "ez_operating", "ez_total_revenue", "ez_total_expenses", "ez_excess_revenue_over_expenses"))
supp_checks.append(reconcile(ez, "ez_balance", "ez_total_assets_eoy", "ez_total_liabilities_eoy", "ez_net_assets_eoy"))
pf = others["IRS990PF"]
supp_checks.append(reconcile(pf, "pf_operating", "pf_total_revenue_books", "pf_total_expenses_books", "pf_excess_revenue_over_expenses"))
supp_checks.append(reconcile(pf, "pf_balance", "pf_total_assets_book_eoy", "pf_total_liabilities_eoy", "pf_net_assets_eoy"))
# PF fair-market-value assets are NOT interchangeable with book assets in the balance sheet identity.
# 990-T tax is NOT a full organization operating expense/revenue measure.
display(pd.DataFrame(supp_checks))

canonical_other = []
for form, df in others.items():
    valid = df.loc[df.key_valid].copy()
    valid["_period_end_sort"] = pd.to_datetime(valid.get("tax_period_end"), errors="coerce")
    valid["_pub_sort"] = pd.to_numeric(valid.get("publication_year"), errors="coerce")
    group_size = valid.groupby(["ein","tax_year"], dropna=False).size().rename("same_form_org_year_filing_count").reset_index()
    valid = valid.sort_values(["_period_end_sort","_pub_sort","filing_id"], ascending=[False,False,False], kind="stable", na_position="last")
    selected = valid.drop_duplicates(["ein","tax_year"]).drop(columns=["_period_end_sort","_pub_sort"])
    selected = selected.merge(group_size, on=["ein","tax_year"], how="left", validate="one_to_one")
    selected["canonical_selection_method"] = "latest_period_end_publication_filing_id_not_verified_amendment"
    selected["canonical_amendment_verified"] = False
    canonical_other.append(selected)
other_canonical = pd.concat(canonical_other, ignore_index=True)
assert not other_canonical.duplicated(["form_type","ein","tax_year"]).any()
print("Separate supplementary canonical rows:", len(other_canonical))
display(other_canonical.groupby("form_type").size().rename("canonical_rows").to_frame())



### 7A. Financial Reconciliation - Root-Cause Investigation

**Investigation Summary**

The financial reconciliation checks identified the following discrepancies:

| Form Type | Eligible Filings | Discrepancies | Rate |
| --- | ---: | ---: | ---: |
| 990-EZ | 458,567 | 9,742 | 2.12% |
| 990-PF | 450,296 | 13,048 | 2.90% |

**Original XML Investigation**

We examined Form 990-PF filing `202533149349102223` and confirmed a discrepancy of **55,227,087 dollars** between reported end-of-year assets minus liabilities and reported net assets.

The discrepancy is present in the original IRS XML and was not introduced by our extraction code for this filing.

**Additional Findings**

Among the 13,048 mismatched 990-PF filings:

- 5,479 reported zero end-of-year assets.
- 11,437 reported zero end-of-year liabilities.
- 6,658 reported net assets greater than reported assets.

These categories overlap.

**Decision**

Preserve original financial values and reconciliation flags. Treat discrepancies as data-quality and investigative signals, not confirmed fraud.

Our positive label represents a self-reported significant diversion of assets, not independently verified fraud.

**Diagnostic Execution**

Lightweight diagnostics run during normal notebook execution. The original XML investigation is optional because scanning the staging files can be slow.

To enable the XML investigation, set `RUN_DEEP_XML_DIAGNOSTICS = True` in the configuration cell.


#### 7A.1 — Balance sheet discrepancy distributions and largest examples

In [ ]:
import pandas as pd
from pathlib import Path

# Project root
ROOT = ROOT  # Reuse the project root detected in Section 1

# Folder containing Notebook 04B outputs
DATA_DIR = ROOT / "data/analytics/other_990_series_features"

# Files and financial fields to check
forms = {
    "IRS990EZ": {
        "file": "irs990ez_filing_features_v1.parquet",
        "assets": "ez_total_assets_eoy",
        "liabilities": "ez_total_liabilities_eoy",
        "net_assets": "ez_net_assets_eoy"
    },
    "IRS990PF": {
        "file": "irs990pf_filing_features_v1.parquet",
        "assets": "pf_total_assets_book_eoy",
        "liabilities": "pf_total_liabilities_eoy",
        "net_assets": "pf_net_assets_eoy"
    }
}

for form, config in forms.items():

    # Load financial data
    df = pd.read_parquet(DATA_DIR / config["file"])

    assets = config["assets"]
    liabilities = config["liabilities"]
    net_assets = config["net_assets"]

    # Only evaluate records with all required values
    required = [assets, liabilities, net_assets]
    x = df.dropna(subset=required).copy()

    # Assets - liabilities should equal net assets
    x["balance_difference"] = (
        x[assets] - x[liabilities] - x[net_assets]
    )

    # Find discrepancies greater than $1
    bad = x[x["balance_difference"].abs() > 1].copy()

    print(f"\n{'=' * 60}")
    print(form)
    print(f"Eligible records: {len(x):,}")
    print(f"Mismatches > $1: {len(bad):,}")
    print(f"Mismatch rate: {len(bad) / len(x) * 100:.2f}%")

    if not bad.empty:

        print("\nDifference distribution:")
        print(
            bad["balance_difference"]
            .abs()
            .describe(percentiles=[0.5, 0.9, 0.99])
        )

        print("\n10 largest discrepancies:")

        bad = bad.sort_values(
            "balance_difference",
            key=lambda s: s.abs(),
            ascending=False
        )

        display(
            bad[
                ["filing_id", "ein", "tax_year"]
                + required
                + ["balance_difference"]
            ].head(10)
        )

#### 7A.2 — Patterns in 990-PF balance sheet discrepancies

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = ROOT  # Reuse the project root detected in Section 1

path = (
    ROOT / "data/analytics/other_990_series_features"
    / "irs990pf_filing_features_v1.parquet"
)

df = pd.read_parquet(path)

# Compare beginning-of-year and end-of-year reconciliation
required = [
    "pf_total_assets_book_eoy",
    "pf_total_liabilities_eoy",
    "pf_net_assets_eoy"
]

x = df.dropna(subset=required).copy()

x["balance_difference"] = (
    x["pf_total_assets_book_eoy"]
    - x["pf_total_liabilities_eoy"]
    - x["pf_net_assets_eoy"]
)

bad = x[x["balance_difference"].abs() > 1].copy()

print("Total 990-PF filings:", f"{len(df):,}")
print("Eligible for balance check:", f"{len(x):,}")
print("Balance mismatches:", f"{len(bad):,}")

print("\nMismatches with zero end-of-year assets:")
print(f"{(bad['pf_total_assets_book_eoy'] == 0).sum():,}")

print("\nMismatches with zero end-of-year liabilities:")
print(f"{(bad['pf_total_liabilities_eoy'] == 0).sum():,}")

print("\nMismatches where net assets exceed reported assets:")
print(
    f"{(bad['pf_net_assets_eoy'] > bad['pf_total_assets_book_eoy']).sum():,}"
)

print("\nMismatches by tax year:")
display(
    bad.groupby("tax_year")
    .agg(
        mismatch_count=("filing_id", "size"),
        median_absolute_difference=(
            "balance_difference",
            lambda s: s.abs().median()
        )
    )
    .reset_index()
    .sort_values("tax_year")
)

#### 7A.3 — Optional original XML source verification

This can be slow because it scans the staging Parquet partitions. It is **off by default**. The findings above document the previously observed source-level discrepancy.

In [ ]:
RUN_DEEP_XML_DIAGNOSTICS = False  # Set True only to repeat the source XML investigation

if RUN_DEEP_XML_DIAGNOSTICS:
    import pandas as pd
    import pyarrow.parquet as pq
    from pathlib import Path
    import json

    ROOT = Path("/Users/julietday/Coding Projects/irs-990-fraud-detection")

    SOURCE = ROOT / "data/staging/other_990_xml_partitions/full"

    # Example of a filing with a large discrepancy
    TARGET_FILING = "202533149349102223"

    def local_name(tag):
        return str(tag).rsplit("}", 1)[-1].rsplit(":", 1)[-1]

    def walk(node, path=()):
        for child in node.get("children", []):
            name = local_name(child.get("tag", ""))
            new_path = path + (name,)
            yield new_path, child
            yield from walk(child, new_path)

    found = False

    for path in sorted(SOURCE.glob("*/irs990pf.parquet")):
        pf = pq.ParquetFile(path)

        for batch in pf.iter_batches(
            batch_size=1000,
            columns=["filing_id", "xml_tree_json"]
        ):
            df = batch.to_pandas()
            match = df[df["filing_id"].astype(str) == TARGET_FILING]

            if match.empty:
                continue

            root = json.loads(match.iloc[0]["xml_tree_json"])

            print("Found filing in:", path.name)

            for field_path, node in walk(root):
                joined = "/".join(field_path)

                if any(term in joined.lower() for term in [
                    "balancesheet",
                    "netassets",
                    "netast",
                    "liabilities",
                    "totalassets"
                ]):
                    if node.get("text") not in (None, ""):
                        print(joined, "=", node["text"])

            found = True
            break

        if found:
            break

    if not found:
        print("Filing not found in staging files.")
else:
    print("Deep XML diagnostics skipped (RUN_DEEP_XML_DIAGNOSTICS=False).")

## 8. Link supplementary returns to the full Form 990 cohort — **counts only**

This link indicates the presence of another return for the same EIN and tax year. It is **not** a financial join and must not be used as a model feature without leakage/filing-availability review. In particular, 990-T is not a separate organization.

In [ ]:
other_key_counts = (other_canonical.groupby(["ein","tax_year","form_type"], observed=True)
                    .size().unstack("form_type", fill_value=0).reset_index())
for form in OTHER:
    if form not in other_key_counts:
        other_key_counts[form] = 0
other_key_counts = other_key_counts.rename(columns={f: f"linked_{f.lower()}_return_count" for f in OTHER})
links = model[["ein","tax_year","filing_id"]].merge(other_key_counts, on=["ein","tax_year"], how="left", validate="one_to_one")
for form in OTHER:
    col = f"linked_{form.lower()}_return_count"
    links[col] = links[col].fillna(0).astype("int8")
assert len(links) == len(model)
print("Supplementary return links in full 990 cohort:")
display(links[[f"linked_{f.lower()}_return_count" for f in OTHER]].sum().rename("matched_organization_years").to_frame())
# Keep link metadata in its own output; do not merge it into model automatically.


## 9. Final QA gates and audit summary

These are **structural** assertions; mismatch counts are surfaced for investigation rather than discarded. Missing or conflicting labels remain excluded only according to Notebook 02’s candidate policy. Verify anomalies before treating the output as modeling-ready.

In [ ]:
assert len(model) == len(candidates)
assert model["filing_id"].is_unique
assert not model.duplicated(["ein","tax_year"]).any()
assert model["ein"].str.fullmatch(r"\d{9}").fillna(False).all()
assert model["tax_year"].notna().all()
assert not other_canonical.duplicated(["form_type","ein","tax_year"]).any()
assert len(links) == len(model)

summary = {
    "notebook": "05_final_dataset_assembly_and_quality_validation",
    "input_candidate_rows": int(len(candidates)),
    "model_rows": int(len(model)),
    "unique_model_eins": int(model.ein.nunique()),
    "positive_disclosures": int(model.fraud_label.eq(1).sum()),
    "negative_disclosures": int(model.fraud_label.eq(0).sum()),
    "missing_disclosure_labels": int(model.fraud_label.isna().sum()),
    "other_filing_counts": {k:int(len(v)) for k,v in others.items()},
    "other_canonical_rows": int(len(other_canonical)),
    "balance_tolerance_dollars": BALANCE_TOLERANCE,
    "full_reconciliation": reconciliation.to_dict("records"),
    "supplement_reconciliation": pd.DataFrame(supp_checks).to_dict("records"),
    "limitations": [
        "Self-reported significant diversion is not verified fraud",
        "Only full Form 990 candidates receive disclosure labels; other forms remain supplementary",
        "Canonical supplementary filing selection is deterministic, not amendment-verified",
        "Source financial figures are not independently audited; mismatch flags are not fraud proof",
        "Do not use label/disclosure-derived fields or post-outcome information as model predictors",
        "Time-aware and EIN-grouped evaluation, leakage checks, and train-only imputation remain for Notebook 06",
        "Missing liabilities are not automatically zero"
    ]
}
print(json.dumps(summary, indent=2, default=str)[:7000])
print("\nQA gates passed. Inspect reconciliation mismatch counts before enabling output writes.")


## 10. Save outputs (explicit opt-in)

Only change `WRITE_OUTPUTS = True` **after** reviewing Sections 4–9. You may set the variable in this cell and run this cell alone, once QA has passed. Files are written atomically where practical; existing outputs are overwritten only when explicitly enabled.

In [ ]:
# Change only after inspecting the QA results:
# WRITE_OUTPUTS = True
if not WRITE_OUTPUTS:
    print("WRITE_OUTPUTS=False: No files written. Review QA, then enable and rerun this cell.")
else:
    assert len(model) == len(candidates) and model.filing_id.is_unique
    outputs = {
        MODEL / "05_form990_modeling_organization_year_v1.parquet": model,
        MODEL / "05_other_series_canonical_v1.parquet": other_canonical,
        MODEL / "05_other_series_link_summary_v1.parquet": links,
    }
    for path, df in outputs.items():
        temp = path.with_name(path.stem + ".tmp.parquet")
        df.to_parquet(temp, index=False)
        temp.replace(path)
        print(f"Saved {len(df):,} rows -> {path}")
    reconciliation.to_csv(REPORT / "05_full990_reconciliation.csv", index=False)
    pd.DataFrame(supp_checks).to_csv(REPORT / "05_supplement_reconciliation.csv", index=False)
    other_counts.to_csv(REPORT / "05_other_series_duplicate_key_audit.csv", index=False)
    (REPORT / "05_quality_manifest.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
    print("Notebook 05 outputs saved. Next: Notebook 06 — leakage-safe feature selection and anomaly detection.")


In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path("/Users/julietday/Coding Projects/irs-990-fraud-detection")
DATA_DIR = ROOT / "data/analytics/modeling"

files = {
    "1. Main Form 990 modeling dataset": "05_form990_modeling_organization_year_v1.parquet",
    "2. Other 990 series canonical dataset": "05_other_series_canonical_v1.parquet",
    "3. Supplementary return link summary": "05_other_series_link_summary_v1.parquet"
}

for name, filename in files.items():
    df = pd.read_parquet(DATA_DIR / filename)

    print(f"\n{'=' * 70}")
    print(name)
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"{'=' * 70}")

    display(df.head(5))

In [ ]:
for name, filename in files.items():
    df = pd.read_parquet(DATA_DIR / filename)

    print(f"\n{name}")
    display(
        pd.DataFrame({
            "column": df.columns,
            "dtype": df.dtypes.astype(str).values
        })
    )

## Interpretation and next notebook

- **The main modeling table contains only eligible full Form 990 organization-years**, with the original selected filing and self-reported diversion disclosure. Other forms are retained separately.
- **Financial reconciliation failures are flagged, not removed**; inspect their distribution and source returns before choosing any exclusion threshold.
- **Do not use a random row split**: the same EIN can appear in multiple years. For predictive claims, evaluate using a temporal holdout and group by EIN where appropriate; avoid information from filings published after the prediction point.
- **Do not equate 990-PF fair-market-value assets with book assets**. 990-T taxable business income is not total nonprofit revenue.
- **No model training or fraud conclusions occur here**. Notebook 06 should define the prediction target and time, explicitly approve features, and report class imbalance and label limitations.
